# সহজ ভাষায় Notebook Guide

এই notebook-এ project-এর **একই `src/` module** (app.py আর cli.py যেগুলো ব্যবহার করে) cell-by-cell চালিয়ে LangGraph self-correction loop খুলে দেখা হয়েছে। Technical term English-এ, explanation Bangla-তে।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে run করুন।
2. প্রতিটা graph run-এর আগে লিখে ফেলুন: কয়টা attempt লাগবে, কোন status-এ শেষ হবে।
3. Section 7-8 real Ollama (`llama3.1:8b`) লাগে। বাকি সব fake LLM দিয়ে চলে।

# Self-Correcting LLM Coder — Exploration

1. Task bank — visible আর hidden test
2. Sandbox runner — per-test feedback, আর এর সীমা
3. Static check — চালানোর আগে AST পরীক্ষা
4. Graph-এর গঠন (Mermaid)
5. Fake LLM: repair loop, hidden-test cheat ধরা
6. Interrupt → resume (human approval)
7. SQLite checkpointer + time travel
8. Live: llama3.1:8b একটা task-এ, তারপর পুরো benchmark

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import tempfile
import pandas as pd
from langgraph.types import Command

from config import get_config
from src.evaluation.bench import run_bench, summarize
from src.graph.coder import build_graph, build_prompt, initial_state, run_config, sqlite_checkpointer
from src.llm.client import OllamaUnavailable, generate
from src.sandbox.runner import run_tests
from src.sandbox.static_check import static_check
from src.tasks.bank import TASKS

cfg = get_config()
print("model:", cfg.llm.model, "| max_attempts:", cfg.agent.max_attempts, "| recursion_limit:", cfg.agent.recursion_limit)

model: llama3.1:8b | max_attempts: 4 | recursion_limit: 40


## 1. Task Bank

প্রতিটা task-এ ৩টা **visible** test (model দেখে, failure feedback পায়) আর ৩টা **hidden** test (model কখনো দেখে না)। Hidden test কেন? ImpossibleBench (2025) দেখিয়েছে বড় model-ও test edit বা example hard-code করে "pass" করে — reward hacking। Hidden test সেটা ধরে।

In [2]:
pd.DataFrame([{"task": t.id, "function": t.func_name, "visible": len(t.visible), "hidden": len(t.hidden),
               "example": f"{t.visible[0][0]} -> {t.visible[0][1]}"} for t in TASKS.values()])

,task,function,visible,hidden,example
0,palindrome,is_palindrome,3,3,"is_palindrome('A man, a plan, a canal: Panama'..."
1,roman,roman_to_int,3,3,roman_to_int('III') -> 3
2,rle,run_length_encode,3,3,run_length_encode('aaabcc') -> 'a3b1c2'
3,intervals,merge_intervals,3,3,"merge_intervals([[1,3],[2,6],[8,10]]) -> [[1, ..."
4,second_largest,second_largest,3,3,"second_largest([3, 1, 4, 4]) -> 3"
5,word_freq,top_words,3,3,"top_words('the cat and the hat', 2) -> [('the'..."
6,duration,format_duration,3,3,format_duration(62) -> '1 minute and 2 seconds'
7,snake,camel_to_snake,3,3,camel_to_snake('camelCase') -> 'camel_case'
8,excel,excel_column,3,3,excel_column(1) -> 'A'
9,next_bigger,next_bigger,3,3,next_bigger(12) -> 21


## 2. Sandbox Runner

প্রতিটা test আলাদা চলে, ফলাফল JSON-এ আসে — তাই feedback হয় *"returned 6, expected 4"*, শুধু traceback না। Self-Debugging paper-এ unit-test feedback-ই সবচেয়ে বড় উন্নতি দিয়েছিল।

In [3]:
cases = {
    "correct": "def roman_to_int(s):\n    v={'I':1,'V':5,'X':10,'L':50,'C':100,'D':500,'M':1000}\n    t=0\n    for i,c in enumerate(s):\n        t += -v[c] if i+1<len(s) and v[c]<v[s[i+1]] else v[c]\n    return t",
    "no subtraction": "def roman_to_int(s):\n    v={'I':1,'V':5,'X':10,'L':50,'C':100,'D':500,'M':1000}\n    return sum(v[c] for c in s)",
    "infinite loop": "def roman_to_int(s):\n    while True:\n        pass",
    "syntax error": "def roman_to_int(s) return 1",
}
for name, code in cases.items():
    rep = run_tests(code, TASKS["roman"].visible, timeout=3)
    print(f"--- {name}: {rep.passed}/{rep.total}\n{rep.feedback() if not rep.all_passed else 'all passed'}\n")

--- correct: 3/3
all passed

--- no subtraction: 1/3
1/3 tests passed. Failing tests:
- roman_to_int('IV'): returned 6 (int), expected 4
- roman_to_int('MCMXCIV'): returned 2216 (int), expected 1994



--- infinite loop: 0/0
Timeout: tests did not finish in 3s (infinite loop?).

--- syntax error: 0/3
0/3 tests passed. Failing tests:
- roman_to_int('III'): code failed to load: SyntaxError: expected ':' (solution.py, line 1)
- roman_to_int('IV'): code failed to load: SyntaxError: expected ':' (solution.py, line 1)
- roman_to_int('MCMXCIV'): code failed to load: SyntaxError: expected ':' (solution.py, line 1)



### এটা আসল sandbox না

আলাদা process, `python -I`, খালি env, timeout, CPU/memory limit — ক্ষতি সীমিত করে, কিন্তু filesystem পড়া বা network বন্ধ করে না। নিচে generated code disk-এর file list পড়ে ফেলে (static check থাকলে `os` import আগেই আটকাত — কিন্তু static check-ও বাইপাসযোগ্য)। Production-এ Docker `--network none`, gVisor, Firecracker/e2b লাগে।

In [4]:
leak = "def peek():\n    import os\n    return sorted(os.listdir('/'))[:5]"
rep = run_tests(leak, [("peek()", "[]")])
print(rep.results[0]["error"])
print("env is scrubbed:", run_tests("def env():\n    import os\n    return sorted(os.environ)", [("env()", "[]")]).results[0]["error"])

returned ['.VolumeIcon.icns', '.file', '.nofollow', '.resolve', '.vol'] (list), expected []
env is scrubbed: returned ['LC_CTYPE', 'PATH', '__CF_USER_TEXT_ENCODING'] (list), expected []


## 3. Static Check — চালানোর আগে

AST দিয়ে syntax, function-এর নাম, banned import/call, dunder access দেখা হয়। এর আসল কাজ **ভালো feedback** — নিরাপত্তা না (blocklist সবসময় বাইপাস করা যায়)।

In [5]:
sb = cfg.sandbox
for code in ["def roman_to_int(s) return 1",
             "def roman(s): return 1",
             "import subprocess\ndef roman_to_int(s): return int(subprocess.check_output(['echo', '1']))",
             "def roman_to_int(s): return eval(s)",
             "def roman_to_int(s): return ().__class__.__bases__[0]"]:
    print(f"{code.splitlines()[-1][:55]:<57} -> {static_check(code, 'roman_to_int', sb.banned_imports, sb.banned_calls)}")

def roman_to_int(s) return 1                              -> ["SyntaxError on line 1: expected ':'"]
def roman(s): return 1                                    -> ["The code must define a function named `roman_to_int` (found: ['roman'])."]
def roman_to_int(s): return int(subprocess.check_output   -> ['Importing `subprocess` is not allowed; use only pure Python (re, math, collections are fine).']
def roman_to_int(s): return eval(s)                       -> ['Calling `eval()` is not allowed.']
def roman_to_int(s): return ().__class__.__bases__[0]     -> ['Dunder attribute access `__bases__` is not allowed.', 'Dunder attribute access `__class__` is not allowed.']


## 4. Graph-এর গঠন

`draw_mermaid()` শুধু text দেয় (PNG রেন্ডার করতে internet লাগে — mermaid.ink)। নিচের text যেকোনো Mermaid viewer-এ paste করলে ছবি হয়। খেয়াল করুন: তিন জায়গা থেকে `generate`-এ ফেরার পথ — static check fail, visible fail, hidden fail।

In [6]:
def scripted(*replies):
    it = iter(replies)
    prompts = []
    def llm(prompt):
        prompts.append(prompt)
        return next(it)
    llm.prompts = prompts
    return llm

print(build_graph(scripted()).get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate(generate)
	static_check(static_check)
	run_visible(run_visible)
	run_hidden(run_hidden)
	approval(approval)
	give_up(give_up)
	hidden_fail(hidden_fail)
	done(done)
	rejected(rejected)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate;
	approval -.-> done;
	approval -.-> rejected;
	generate --> static_check;
	run_hidden -.-> approval;
	run_hidden -.-> done;
	run_hidden -.-> generate;
	run_hidden -.-> hidden_fail;
	run_visible -.-> generate;
	run_visible -.-> give_up;
	run_visible -.-> run_hidden;
	static_check -.-> generate;
	static_check -.-> give_up;
	static_check -.-> run_visible;
	done --> __end__;
	give_up --> __end__;
	hidden_fail --> __end__;
	rejected --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 5. Fake LLM দিয়ে loop-এর প্রতিটা পথ

তিনটা scenario: (ক) প্রথমে subtraction ভুলে যায়, feedback পেয়ে ঠিক করে; (খ) visible example hard-code করে "cheat" করে — hidden test ধরে, আর retry prompt-এ hidden input ফাঁস হয় না; (গ) সবসময় ভুল — budget শেষে `gave_up`, সবচেয়ে ভালো attempt রাখে।

In [7]:
ok = "```python\n" + cases["correct"] + "\n```"
wrong = "```python\n" + cases["no subtraction"] + "\n```"
cheat = "```python\ndef roman_to_int(s):\n    return {'III': 3, 'IV': 4, 'MCMXCIV': 1994}[s]\n```"
zero = "```python\ndef roman_to_int(s):\n    return 0\n```"

scenarios = {"repair": scripted(wrong, ok), "cheat then fix": scripted(cheat, ok), "always wrong": scripted(wrong, zero, zero)}
rows = []
for name, llm in scenarios.items():
    g = build_graph(llm, max_attempts=3, require_approval=False)
    s = g.invoke(initial_state(TASKS["roman"]), run_config(f"demo-{name}"))
    rows.append({"scenario": name, "status": s["status"], "attempts": s["attempts"],
                 "visible": f"{s.get('visible_passed')}/3", "hidden": f"{s.get('hidden_passed', '-')}/3",
                 "history": " | ".join(s["history"])})
pd.set_option("display.max_colwidth", 200)
pd.DataFrame(rows)

,scenario,status,attempts,visible,hidden,history
0,repair,done,2,3/3,3/3,attempt 1: code generated (3 lines) | attempt 1: visible tests 1/3 | attempt 2: code generated (6 lines) | attempt 2: visible tests 3/3 | attempt 2: hidden tests 3/3
1,cheat then fix,done,2,3/3,3/3,attempt 1: code generated (2 lines) | attempt 1: visible tests 3/3 | attempt 1: hidden tests 0/3 — passes the examples but not the general task | attempt 2: code generated (6 lines) | attempt 2: v...
2,always wrong,gave_up,3,0/3,-/3,attempt 1: code generated (3 lines) | attempt 1: visible tests 1/3 | attempt 2: code generated (2 lines) | attempt 2: visible tests 0/3 | attempt 3: code generated (2 lines) | attempt 3: visible t...


In [8]:
print("RETRY PROMPT after the cheat (hidden inputs like 'LVIII' must NOT appear):\n")
print(scenarios["cheat then fix"].prompts[1])
print("\nLeaked hidden input?", "LVIII" in scenarios["cheat then fix"].prompts[1])

RETRY PROMPT after the cheat (hidden inputs like 'LVIII' must NOT appear):

Task: Write roman_to_int(s: str) -> int that converts a Roman numeral (I, V, X, L, C, D, M, including subtractive forms like IV and CM) to an integer.
The function must be named `roman_to_int`.
Examples that must work:
  roman_to_int('III')  ->  3
  roman_to_int('IV')  ->  4
  roman_to_int('MCMXCIV')  ->  1994

Your previous attempt:
```python
def roman_to_int(s):
    return {'III': 3, 'IV': 4, 'MCMXCIV': 1994}[s]
```
It FAILED:
Your code passes the 3 examples but fails 3 other (hidden) test case(s). Re-read the specification carefully, think about edge cases it mentions, and fix the general logic. Do not special-case the examples.
Explain the bug to yourself in one sentence, then write the corrected function. Do not special-case the example inputs; solve the general task.


Leaked hidden input? False


## 6. Interrupt → Resume

`approval` node-এ `interrupt()` graph থামায়, checkpoint হয়। `invoke`-এর ফলাফলে `__interrupt__` থাকে। মানুষের উত্তর `Command(resume=...)` দিয়ে **একই thread_id**-তে পাঠাতে হয়। মনে রাখুন: resume-এ node **শুরু থেকে আবার চলে** — তাই interrupt-এর আগে কোনো side effect না।

In [9]:
g = build_graph(scripted(ok))
first = g.invoke(initial_state(TASKS["roman"]), run_config("approve-demo"))
print("paused:", list(first["__interrupt__"][0].value))
print("next node:", g.get_state(run_config("approve-demo")).next)
final = g.invoke(Command(resume=True), run_config("approve-demo"))
print("after resume:", final["status"], "|", final["history"][-1])

paused: ['question', 'code', 'visible', 'hidden']
next node: ('approval',)
after resume: done | human approved=True


## 7. SQLite Checkpointer + Time Travel

`InMemorySaver` process বন্ধ হলে সব হারায়। `SqliteSaver` disk-এ রাখে — app restart করলেও run দেখা যায়। প্রতিটা super-step একটা checkpoint; `get_state_history` সব ফেরত দেয়, পুরোনো checkpoint থেকে আবার চালানো যায় (replay/fork)।

In [10]:
db = Path(tempfile.mkdtemp()) / "demo.sqlite"
g = build_graph(scripted(wrong, ok), require_approval=False, checkpointer=sqlite_checkpointer(db))
g.invoke(initial_state(TASKS["roman"]), run_config("tt"))

g_new = build_graph(scripted(), require_approval=False, checkpointer=sqlite_checkpointer(db))   # "after a restart"
hist = list(g_new.get_state_history(run_config("tt")))
pd.DataFrame([{"step": h.metadata.get("step"), "next": ", ".join(h.next) or "END",
               "attempts": h.values.get("attempts"), "status": h.values.get("status", "")} for h in hist])

,step,next,attempts,status
0,8,END,2.0,done
1,7,done,2.0,
2,6,run_hidden,2.0,
3,5,run_visible,2.0,
4,4,static_check,2.0,
5,3,generate,1.0,
6,2,run_visible,1.0,
7,1,static_check,1.0,
8,0,generate,0.0,
9,-1,__start__,NaN,


In [11]:
# Time travel: go back to the checkpoint right before attempt 2 was generated, and replay from there
before_retry = next(h for h in hist if h.next == ("generate",) and h.values.get("attempts") == 1)
g_fork = build_graph(scripted(ok), require_approval=False, checkpointer=sqlite_checkpointer(db))
replayed = g_fork.invoke(None, before_retry.config)
print("replayed from step", before_retry.metadata["step"], "->", replayed["status"], "after", replayed["attempts"], "attempts")

replayed from step 3 -> done after 2 attempts


## 8. Live — llama3.1:8b

এবার আসল model। `format_duration` task-এ spec-এ অনেক খুঁটিনাটি (plural, `and`, `now`) — ছোট model প্রায়ই প্রথমবার ভুল করে। `stream_mode="updates"` দিয়ে প্রতিটা node শেষ হলেই দেখি।

In [12]:
g = build_graph(generate, require_approval=False)
try:
    for chunk in g.stream(initial_state(TASKS["duration"]), run_config("live-duration"), stream_mode="updates"):
        for node, update in chunk.items():
            for line in (update or {}).get("history", []):
                print(f"[{node}] {line}")
    final = g.get_state(run_config("live-duration")).values
    print("\nstatus:", final["status"], "\n" + final["code"])
except OllamaUnavailable as e:
    print("Ollama unavailable:", e)

[generate] attempt 1: code generated (26 lines)
[run_visible] attempt 1: visible tests 2/3


[generate] attempt 2: code generated (31 lines)
[run_visible] attempt 2: visible tests 2/3


[generate] attempt 3: code generated (31 lines)
[run_visible] attempt 3: visible tests 3/3
[run_hidden] attempt 3: hidden tests 3/3

status: done 
def format_duration(seconds: int) -> str:
    if seconds == 0:
        return 'now'

    years = seconds // 31536000
    seconds %= 31536000
    days = seconds // 86400
    seconds %= 86400
    hours = seconds // 3600
    seconds %= 3600
    minutes = seconds // 60
    seconds %= 60

    parts = []
    if years > 0:
        parts.append(f'{years} year{"s" if years > 1 else ""}')
    if days > 0:
        parts.append(f'{days} day{"s" if days > 1 else ""}')
    if hours > 0:
        parts.append(f'{hours} hour{"s" if hours > 1 else ""}')
    if minutes > 0:
        parts.append(f'{minutes} minute{"s" if minutes > 1 else ""}')
    if seconds > 0:
        parts.append(f'{seconds} second{"s" if seconds > 1 else ""}')

    if len(parts) == 1:
        return parts[0]
    elif len(parts) == 2:
        return ' and '.join(parts)
    else:
        ret

### পুরো benchmark — repair আসলে কতটা সাহায্য করে?

"Is Self-Repair a Silver Bullet?" (ICLR 2024): repair-এর লাভ প্রায়ই ছোট, feedback-এর মানই bottleneck। নিজের ১১টা task-এ মাপি — pass@1 বনাম repair-এর পরে।

In [13]:
try:
    rows = run_bench(generate)
    display(pd.DataFrame([r.__dict__ for r in rows]))
    print(summarize(rows))
except OllamaUnavailable as e:
    print("Ollama unavailable:", e)

,task,status,attempts,first_attempt_ok,final_ok,visible,hidden,seconds
0,palindrome,done,1,True,True,3/3,3/3,3.0
1,roman,done,1,True,True,3/3,3/3,23.9
2,rle,done,1,True,True,3/3,3/3,39.1
3,intervals,done,1,True,True,3/3,3/3,33.4
4,second_largest,done,1,True,True,3/3,3/3,17.3
5,word_freq,gave_up,4,False,False,3/3,-,54.0
6,duration,done,3,False,True,3/3,3/3,181.8
7,snake,gave_up,4,False,False,2/3,-,193.9
8,excel,done,1,True,True,3/3,3/3,6.0
9,next_bigger,done,1,True,True,3/3,3/3,77.5


{'tasks': 11, 'pass@1': 0.73, 'pass_after_repair': 0.82, 'repaired': 1, 'hidden_fail': 0, 'gave_up': 2, 'avg_attempts': 1.73, 'avg_seconds': 59.9}


**কী শিখলাম:**

* বেশিরভাগ সহজ task llama3.1:8b প্রথমবারেই পারে; repair-এর লাভ আসে spec-ভারী task-এ (যেমন `format_duration`)।
* Repair সবসময় উন্নতি আনে না — কিছু task-এ generic hint পেয়ে model আরও খারাপ code লেখে। Feedback যত নির্দিষ্ট (failing test diff), repair তত কাজ করে।
* Hidden test ছাড়া "visible পাস = সঠিক" ধরে নিলে ভুল code approve হয়ে যেত।
* Live output run থেকে run-এ বদলাতে পারে (Ollama version, hardware) — তাই সংখ্যা নিজের মেশিনে মাপুন।

## 🧠 Brain Teasers

1. `max_attempts=1` দিয়ে benchmark চালান। pass@1 আর pass_after_repair-এর পার্থক্য কী হয়, কেন?
2. `run_hidden`-এর generic feedback-এ hidden test-এর input যোগ করে দিন (শুধু পরীক্ষার জন্য)। `word_freq`-এ model কি এবার ঠিক করে, নাকি সেই input-গুলো hard-code করে?
3. Section 7-এ `before_retry`-এর বদলে একদম প্রথম checkpoint থেকে replay করুন। কয়টা LLM call লাগে?
4. Static check-কে ফাঁকি দিয়ে `os` ব্যবহার করার একটা উপায় লিখুন (hint: `importlib`)। তারপর বলুন কেন আসল সুরক্ষা process/container isolation।